In [1]:
# ================= FIXED ADQ QAT | CIFAR-100 | RESNET18 =================
!pip install torch torchvision timm tqdm --quiet

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm
import torch.ao.quantization as quant
from timm.utils import ModelEma

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 100
WARMUP_EPOCHS = 50   # 🔥 IMPORTANT CHANGE
BATCH_SIZE = 128

# ---------------- DATA ----------------
MEAN = [0.5071, 0.4867, 0.4408]
STD  = [0.2675, 0.2565, 0.2761]

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),  # 🔥 stronger aug
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = torchvision.datasets.CIFAR100("./data", train=True, download=True, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR100("./data", train=False, download=True, transform=test_tf)

train_ld = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_ld  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# ---------------- MODEL ----------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model("resnet18", pretrained=True, num_classes=100)

        base.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        base.maxpool = nn.Identity()

        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()

        self.stem = nn.Sequential(base.conv1, base.bn1, base.act1)

        self.layer1 = base.layer1
        self.layer2 = base.layer2

        # 🔥 FP32 critical layers
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.pool = base.global_pool
        self.fc = nn.Sequential(
            nn.Dropout(0.3),   # 🔥 needed for CIFAR-100
            nn.Linear(512, 100)
        )

    def forward(self, x):
        x = self.stem(x)

        x = self.quant(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.dequant(x)

        x = self.layer3(x)
        x = self.layer4(x)

        x = self.pool(x)
        return self.fc(x)

model = Net().to(device)
ema = ModelEma(model, decay=0.999)

# ---------------- OPTIM ----------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # 🔥 important

optimizer = optim.SGD(
    model.parameters(),
    lr=0.05,            # 🔥 higher LR
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ---------------- EVAL ----------------
@torch.no_grad()
def evaluate(m):
    m.eval()
    correct = total = 0
    for x, y in test_ld:
        x, y = x.to(device), y.to(device)
        out = m(x)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

# ---------------- TRAIN ----------------
best_acc = 0
qat_enabled = False

for epoch in range(EPOCHS):
    model.train()

    loop = tqdm(train_ld, desc=f"Epoch {epoch+1}")
    for x, y in loop:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = criterion(out, y)

        # ✅ SAFE ADQ (only after QAT)
        if qat_enabled:
            commitment_loss = 0

            for m in model.modules():
                if isinstance(m, (nn.Conv2d, nn.Linear)):
                    w = m.weight
                    scale = w.abs().max().detach() + 1e-8
                    w_q = torch.clamp(w / scale, -1, 1)

                    commitment_loss += ((w - w_q.detach()) ** 2).mean()

            loss += 0.001 * commitment_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ema.update(model)

    scheduler.step()

    acc = evaluate(ema.ema)
    print(f"Acc: {acc:.4f}")

    # -------- ENABLE QAT --------
    if epoch + 1 == WARMUP_EPOCHS and not qat_enabled:
        print(">>> ENABLE QAT")

        model.qconfig = quant.get_default_qat_qconfig("fbgemm")
        quant.prepare_qat(model, inplace=True)

        for g in optimizer.param_groups:
            g["lr"] = 5e-4

        qat_enabled = True

    if acc > best_acc:
        best_acc = acc
        torch.save(ema.ema.state_dict(), "best_cifar100.pth")

# ---------------- CONVERT ----------------
model.cpu().eval()
quant.convert(model, inplace=True)

print("FINAL ACC:", best_acc)

100%|██████████| 169M/169M [00:13<00:00, 12.3MB/s]


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 391/391 [00:40<00:00,  9.62it/s]


Acc: 0.0244


Epoch 2: 100%|██████████| 391/391 [00:43<00:00,  8.97it/s]


...

Acc: 0.8062


Epoch 100: 100%|██████████| 391/391 [00:57<00:00,  6.85it/s]


Acc: 0.8067


/tmp/ipykernel_23/4257756692.py:169: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant.convert(model, inplace=True)


FINAL ACC: 0.8127
